# Capstone Project 3: What Predicts Life Expectancy? — A Regression Capstone

The first two capstones were both classification. This one is **regression**, using the
Gapminder country-year dataset already present at the repo root of
`3. EDA and Visualization/gapminder.csv` — the same dataset behind Hans Rosling's
famous "the wealth and health of nations" animation.

**The question:** how well can a country's life expectancy be predicted from its
economic and public-health indicators, and which indicator matters most?

### Workflow

1. Load and explore
2. Correlation analysis (stats Notebook 5) — beware confounding
3. Train/test split, baseline, linear regression, random forest regressor
4. Diagnostics: residual plots, not just R²
5. Feature importance and a sanity check against domain knowledge

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
CV = KFold(5, shuffle=True, random_state=0)

## 1. Load and explore

In [ ]:
df = pd.read_csv("../3. EDA and Visualization/gapminder.csv")
print(df.shape)
df.head()

In [ ]:
print(df.describe().round(2))
print("\nmissing values:\n", df.isnull().sum())
print("\nregions:", df["Region"].unique())

## 2. Correlation — and a confounder to watch for

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0,
           square=True, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()

life_corr = corr["life"].drop("life").sort_values(key=abs, ascending=False)
print(life_corr)

In [ ]:
# child_mortality correlates most strongly with life -- but so does GDP, and GDP and
# child_mortality are themselves correlated (richer countries have lower child
# mortality). Following the discipline from stats Notebook 5, check the PARTIAL
# correlation of GDP with life, controlling for child_mortality, before concluding
# GDP has an independent effect.
def partial_corr(x, y, z):
    res_x = x - np.polyval(np.polyfit(z, x, 1), z)
    res_y = y - np.polyval(np.polyfit(z, y, 1), z)
    return stats.pearsonr(res_x, res_y)

valid = df.dropna(subset=["GDP", "life", "child_mortality"])
r_raw, p_raw = stats.pearsonr(valid["GDP"], valid["life"])
r_partial, p_partial = partial_corr(valid["GDP"].values, valid["life"].values,
                                    valid["child_mortality"].values)

print(f"Raw correlation(GDP, life)                        : {r_raw:+.4f} (p={p_raw:.2e})")
print(f"Partial correlation(GDP, life | child_mortality)  : {r_partial:+.4f} (p={p_partial:.2e})")
print()
print("GDP's association with life expectancy shrinks substantially once child")
print("mortality is controlled for -- a lot of GDP's apparent effect is really routing")
print("through public health infrastructure that both wealth and life expectancy share.")

## 3. Train/test split and models

In [ ]:
feature_cols = ["population", "fertility", "HIV", "CO2", "BMI_male", "GDP",
                "BMI_female", "child_mortality"]
data = df.dropna(subset=feature_cols + ["life"]).copy()

X = data[feature_cols]
y = data["life"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"train: {len(y_train)}, test: {len(y_test)}")

In [ ]:
models = {
    "baseline (predict mean)": DummyRegressor(),
    "linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge (tuned)": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 50))),
    "random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0),
}

print(f"{'model':<26}{'CV RMSE':>10}{'test RMSE':>11}{'test R2':>9}")
for name, model in models.items():
    cv_rmse = -cross_val_score(model, X_train, y_train, cv=CV,
                               scoring="neg_root_mean_squared_error").mean()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, pred))
    test_r2 = r2_score(y_test, pred)
    print(f"{name:<26}{cv_rmse:>10.3f}{test_rmse:>11.3f}{test_r2:>9.4f}")

## 4. Residual diagnostics

R² alone can hide a badly-specified model (Anscombe's quartet, stats Notebook 5) — the
residual plot is what actually tells you whether the errors are structureless or
whether the model is systematically wrong somewhere.

In [ ]:
best_model = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0).fit(X_train, y_train)
pred = best_model.predict(X_test)
resid = y_test - pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].scatter(y_test, pred, s=25, alpha=0.7, color="steelblue")
lims = [y_test.min(), y_test.max()]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("actual life expectancy"); axes[0].set_ylabel("predicted")
axes[0].set_title(f"Predicted vs actual (R2={r2_score(y_test, pred):.3f})")

axes[1].scatter(pred, resid, s=25, alpha=0.7, color="steelblue")
axes[1].axhline(0, color="crimson", lw=1.5)
axes[1].set_xlabel("predicted"); axes[1].set_ylabel("residual")
axes[1].set_title("Residuals vs predicted")

axes[2].hist(resid, bins=20, color="steelblue", edgecolor="white")
axes[2].set_title(f"Residual distribution (mean={resid.mean():.2f})")
plt.tight_layout()
plt.show()

worst = data.loc[y_test.index].assign(predicted=pred, residual=resid.values).nlargest(5, "residual", keep="all")
print("\nCountries the model most UNDER-predicts (actual >> predicted):")
print(worst[["Region", "life", "predicted", "residual"]].round(2).to_string(index=False) if "Region" in worst else worst[["life","predicted","residual"]])

## 5. Feature importance

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=30, random_state=0,
                              scoring="r2")
importance = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("Permutation importance (drop in R2)")
plt.tight_layout()
plt.show()
print(importance.round(4))

print()
print("Sanity check against domain knowledge: child_mortality dominating is exactly")
print("what public-health research would predict -- it is close to a direct proxy for")
print("healthcare access and nutrition, the two biggest levers on life expectancy at a")
print("population level. A model where GDP alone dominated, with child_mortality barely")
print("registering, would be a signal to go back and check for a data or leakage bug.")